# Reranking in RAG

**Module:** 03 — Reranking

Wire reranking into the full RAG loop: retrieve, filter, rerank, pack, generate, cite—and know the failure modes when any stage drifts.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Place rerank in a production RAG sequence diagram
- Pack context from reranked hits with citations
- Detect retrieval vs rerank vs generation failures
- Set N/k under token and latency budgets
- Add evaluation hooks around the rerank stage


## Full Pipeline Steps

**Definition.** A **RAG + rerank pipeline** typically: authorize → retrieve N → metadata filter → rerank → diversity/pack → generate → cite/validate.

**Why it matters.** Most 'hallucinations' are packing the wrong chunk. Rerank is the last chance to fix order before tokens are spent in the LLM.

**How it works.** Keep stage contracts explicit; pass structured `Hit` objects, not loose strings.

**Intuition.** Assembly line with QC before the box is sealed (the prompt).

**Common pitfalls.**
- Packing by stage1 after computing stage2
- Reranking after the LLM (too late)
- Dropping citations when reordering
- One global N for all query classes

**When to use.** Any RAG system where top-k evidence quality drives answer quality.

```mermaid
flowchart TB
  Q[Query] --> ACL[ACL / tenant filter]
  ACL --> RET[Retrieve N]
  RET --> RR[Rerank]
  RR --> PACK[Pack + cite]
  PACK --> LLM[Generate]
  LLM --> OUT[Answer + citations]
```

### Failure modes

| Symptom | Likely stage | Check |
|---------|--------------|-------|
| Wrong topic entirely | Retrieve | Recall@N |
| Right doc in list, wrong in prompt | Rerank/pack | nDCG@k, pack logs |
| Right evidence, wrong claim | Generation | Faithfulness tests |
| Intermittent wrong tenant | ACL/filter | Security tests |


In [ ]:
# Demo 1 — end-to-end toy pipeline
from dataclasses import dataclass

@dataclass
class Hit:
    id: str
    text: str
    score: float

CORPUS = [
    Hit("c1", "Refunds within 60 days of purchase.", 0.0),
    Hit("c2", "Shipments arrive in 3–5 business days.", 0.0),
    Hit("c3", "Reset password via email link.", 0.0),
    Hit("c4", "Refunds unavailable on clearance items.", 0.0),
]

def retrieve(q, n=4):
    scored = []
    for h in CORPUS:
        s = len(set(q.lower().split()) & set(h.text.lower().split()))
        scored.append(Hit(h.id, h.text, float(s)))
    return sorted(scored, key=lambda h: h.score, reverse=True)[:n]

def rerank(q, hits):
    def ce(h):
        return h.score + (0.5 if "refund" in q.lower() and "Refund" in h.text else 0.0)
    return sorted(hits, key=ce, reverse=True)

def pack(hits, k=2):
    return "\n".join(f"[{h.id}] {h.text}" for h in hits[:k])

q = "What is the refund window?"
ctx = pack(rerank(q, retrieve(q)))
print(ctx)


In [ ]:
# Demo 2 — context packing after rerank
def pack_budget(hits, max_chars=200):
    out, used = [], 0
    for h in hits:
        chunk = f"[{h.id}] {h.text}\n"
        if used + len(chunk) > max_chars:
            break
        out.append(chunk)
        used += len(chunk)
    return "".join(out), used

from dataclasses import dataclass
@dataclass
class Hit:
    id: str
    text: str
hits = [Hit("a", "x"*80), Hit("b", "y"*80), Hit("c", "z"*80)]
text, used = pack_budget(hits, 120)
print(used, text.replace("\n", " | "))


In [ ]:
# Demo 3 — failure mode classifier
def classify(miss_in_retrieve, bad_order, bad_gen):
    if miss_in_retrieve:
        return "retrieval_recall"
    if bad_order:
        return "rerank_or_pack"
    if bad_gen:
        return "generation"
    return "ok"

print(classify(True, False, False))
print(classify(False, True, False))


In [ ]:
# Demo 4 — grounded chat request shape
import json
YOUR_API_KEY = "YOUR_API_KEY"
req = {
  "model": "gpt-4.1-mini",
  "temperature": 0.2,
  "messages": [
    {"role": "system", "content": "Answer ONLY from CONTEXT. Cite [id]."},
    {"role": "user", "content": "CONTEXT:\n[c1] Refunds within 60 days.\n\nQ: refund window?"},
  ],
}
print(json.dumps(req, indent=2)[:500])
print("key", YOUR_API_KEY[:8] + "...")


### Try it yourself — Full Pipeline Steps

1. Draw your prod sequence including ACL filters.
2. Write one eval query that should refuse when hits are irrelevant.


## Context packing & citations

**Definition.** After rerank, **packing** selects and formats evidence under a token budget, preserving IDs for citations.

**Why it matters.** The LLM can only use what you pack; citations make audits and UX possible.

**How it works.** Walk reranked list; add chunks until budget; optionally diversify; keep stable IDs.

**Intuition.** A suitcase: fold the best clothes first; leave the rest.

**Common pitfalls.**
- Deduping away the only supporting chunk
- Stripping IDs so citations break
- Stuffing low-score chunks 'just in case' past the budget

**When to use.** Every RAG answer path that claims grounding.


In [ ]:
# Demo 1 — citation map
packed = [("c1", "Refunds within 60 days."), ("c4", "Not on clearance.")]
answer = "You have 60 days [c1]. Clearance is excluded [c4]."
print(answer)
print("cited ids", [p[0] for p in packed if f"[{p[0]}]" in answer])


In [ ]:
# Demo 2 — diversity: avoid near-duplicate chunks
def jaccard(a, b):
    A, B = set(a.split()), set(b.split())
    return len(A & B) / (len(A | B) + 1e-9)

selected = []
for text in ["refund 60 days", "refunds within 60 days of purchase", "shipping 3 days"]:
    if all(jaccard(text, s) < 0.5 for s in selected):
        selected.append(text)
print(selected)


In [ ]:
# Demo 3 — token budget estimate
def est_tokens(s):
    return max(1, len(s) // 4)

chunks = ["[" + "x"*100 + "]" for _ in range(10)]
budget, used, kept = 300, 0, []
for c in chunks:
    t = est_tokens(c)
    if used + t > budget:
        break
    kept.append(c); used += t
print("kept", len(kept), "used_tok_est", used)


### Try it yourself — Context packing & citations

1. Define a citation format for your frontend.
2. When would you pack k=1 vs k=8?


## Eval hooks around rerank

**Definition.** Instrument offline metrics **before and after** rerank so you know the stage's lift.

**Why it matters.** Without stage-level metrics, teams 'fix RAG' randomly and thrash prompts.

**How it works.** Log ranked IDs at retrieve and rerank; compute ΔnDCG; slice by query class.

**Intuition.** A/B the sieve, not the whole factory, when possible.

**Common pitfalls.**
- Only measuring final answer thumbs
- Tiny eval that overfits to three FAQs

**When to use.** Before/after every reranker or N change.


In [ ]:
# Demo 1 — delta metric
def ndcg_proxy(ranked, gold_first):
    return 1.0 if ranked and ranked[0] == gold_first else 0.0

before, after = ["d9", "d1"], ["d1", "d9"]
print("lift", ndcg_proxy(after, "d1") - ndcg_proxy(before, "d1"))


In [ ]:
# Demo 2 — query class slices
from collections import defaultdict
rows = [("paraphrase", 0.1), ("paraphrase", 0.3), ("sku", 0.0), ("sku", 0.05)]
agg = defaultdict(list)
for k, v in rows:
    agg[k].append(v)
print({k: sum(v)/len(v) for k, v in agg.items()})


In [ ]:
# Demo 3 — gate
lift, latency_ms, budget_ms = 0.08, 140, 200
ship = lift >= 0.05 and latency_ms <= budget_ms
print("ship" if ship else "no-ship")


### Try it yourself — Eval hooks around rerank

1. Propose a ship gate for your service (metric + latency).
2. List three query classes to slice in eval.


## Glossary

- **packing**: Fitting evidence into a token budget
- **citation**: Stable ID linking answer spans to chunks
- **stage lift**: Metric gain attributable to rerank


### Workshop drill — Reranking in RAG (1)

Restate each major section heading as one exam-ready sentence.


In [ ]:
# Workshop drill 1 — Reranking in RAG
headings = ['Full Pipeline Steps', 'Context packing & citations', 'Eval hooks around rerank']
for h in headings:
    print('-', h, '→', '...')


### Workshop drill — Reranking in RAG (2)

Sketch a latency budget: first-stage ms + rerank (N candidates × cost) + LLM.


In [ ]:
# Workshop drill 2 — Reranking in RAG
first_ms, per_pair_ms, n, llm_ms = 40, 3, 50, 800
print('total_ms', first_ms + n*per_pair_ms + llm_ms)
print('rerank_share', round(n*per_pair_ms/(first_ms+n*per_pair_ms+llm_ms), 3))


### Workshop drill — Reranking in RAG (3)

Design an offline metric slice: 5 queries with graded relevance labels.


In [ ]:
# Workshop drill 3 — Reranking in RAG
eval_set = [{'q':'...','docs':{'d1':2,'d2':1,'d3':0}}]
print('n_queries', len(eval_set))
print('TODO: fill real labels')


### Workshop drill — Reranking in RAG (4)

Write a go/no-go checklist for shipping a reranker in RAG.


In [ ]:
# Workshop drill 4 — Reranking in RAG
for c in ['latency_p95','nDCG@10','cost/1k','cache_hit','fallback']:
    print(f'[ ] {c}')


### Workshop drill — Reranking in RAG (5)

Compare bi-encoder vs cross-encoder in a small table (fill TODOs).


In [ ]:
# Workshop drill 5 — Reranking in RAG
print('| axis | bi | cross |')
print('|------|----|-------|')
print('| latency | TODO | TODO |')
print('| precision | TODO | TODO |')


## Summary & Key Takeaways

- Rerank sits after retrieve/filter and before pack/generate
- Pack from reranked order; keep citation IDs
- Classify failures by stage before changing models
- Measure lift from rerank, not only end-to-end vibes

### Practice

Add retrieve-vs-rerank nDCG logging to a toy pipeline on 10 queries.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
